##Vector Indexing & Retrieval Baseline

Step 1: Mount Drive + set paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/ifixit_dump"
CHUNKS_PATH = f"{BASE}/chunks.jsonl"

OUT_DIR = f"{BASE}/faiss_bge_small"

Mounted at /content/drive




Step 2: Install Dependencies

In [ ]:
!pip -q install faiss-cpu sentence-transformers tqdm orjson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.5 MB/s eta 0:00:00


Step 3: Build FAISS index (batched, saves to Drive)



In [ ]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = SentenceTransformer(MODEL_NAME, device=device)

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 2))

Tesla T4
VRAM GB: 15.64


Enable FP16 (mixed precision) during encoding - SentenceTransformers supports it via model config by putting the model in half precision.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"
model = SentenceTransformer(MODEL_NAME, device="cuda")
model = model.half()  # fp16 for speed on T4

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



*   BATCH_SIZE = 512
*   Best-performing cell for T4, using this one (it includes FP16 + progress speed)



In [ ]:
import os, time
import numpy as np
import faiss
import orjson
import torch
from sentence_transformers import SentenceTransformer

BASE = "/content/drive/MyDrive/ifixit_dump"
CHUNKS_PATH = f"{BASE}/chunks.jsonl"
OUT_DIR = f"{BASE}/faiss_bge_small"
os.makedirs(OUT_DIR, exist_ok=True)

INDEX_PATH = f"{OUT_DIR}/index.faiss"
META_PATH  = f"{OUT_DIR}/meta.jsonl"

BATCH_SIZE = 512

print("GPU:", torch.cuda.get_device_name(0))
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda").half()
dim = model.get_sentence_embedding_dimension()
index = faiss.IndexFlatIP(dim)

added = 0
t0 = time.time()
last_t = time.time()
last_n = 0

with open(CHUNKS_PATH, "rb") as fin, open(META_PATH, "wb") as meta_out:
    batch_texts, batch_meta = [], []

    for line in fin:
        rec = orjson.loads(line)
        txt = rec.get("text", "")
        if not txt:
            continue

        batch_texts.append(txt)
        batch_meta.append({
            "chunk_id": rec.get("chunk_id"),
            "guide_id": rec.get("guide_id"),
            "chunk_index": rec.get("chunk_index"),
            "title": rec.get("title"),
            "url": rec.get("url"),
            "device": rec.get("device"),
            "tools": rec.get("tools"),
            "parts": rec.get("parts"),
            "preview": txt[:220],
        })

        if len(batch_texts) >= BATCH_SIZE:
            emb = model.encode(
                batch_texts,
                batch_size=BATCH_SIZE,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            ).astype(np.float32)

            index.add(emb)
            for m in batch_meta:
                meta_out.write(orjson.dumps(m) + b"\n")

            added += len(batch_texts)
            batch_texts, batch_meta = [], []

            if added % 5000 == 0:
                now = time.time()
                rate = (added - last_n) / (now - last_t + 1e-9)
                print(f"Indexed {added} chunks | {rate:.1f} chunks/sec")
                last_t, last_n = now, added

    if batch_texts:
        emb = model.encode(
            batch_texts,
            batch_size=BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)
        index.add(emb)
        for m in batch_meta:
            meta_out.write(orjson.dumps(m) + b"\n")
        added += len(batch_texts)

faiss.write_index(index, INDEX_PATH)

print("DONE")
print("Total indexed:", index.ntotal)
print("Minutes:", round((time.time()-t0)/60, 2))
print("Saved:", INDEX_PATH)
print("Saved:", META_PATH)

GPU: Tesla T4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DONE
Total indexed: 165523
Minutes: 9.56
Saved: /content/drive/MyDrive/ifixit_dump/faiss_bge_small/index.faiss
Saved: /content/drive/MyDrive/ifixit_dump/faiss_bge_small/meta.jsonl


Step 4 Load index + run simple retrieval tests

In [ ]:
import faiss, orjson, numpy as np
from sentence_transformers import SentenceTransformer
import torch

INDEX_PATH = "/content/drive/MyDrive/ifixit_dump/faiss_bge_small/index.faiss"
META_PATH  = "/content/drive/MyDrive/ifixit_dump/faiss_bge_small/meta.jsonl"
MODEL_NAME = "BAAI/bge-small-en-v1.5"

index = faiss.read_index(INDEX_PATH)
model = SentenceTransformer(MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu").half()

meta = []
with open(META_PATH, "rb") as f:
    for line in f:
        meta.append(orjson.loads(line))

print("Vectors:", index.ntotal, "Meta:", len(meta))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectors: 165523 Meta: 165523


Search helpers

In [ ]:
import numpy as np

def search(query: str, top_k: int = 5):
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q, top_k)
    out = []
    for s, idx in zip(scores[0], ids[0]):
        m = meta[idx]
        out.append((float(s), m))
    return out

def show(results):
    for i, (score, m) in enumerate(results, 1):
        print(f"\n#{i} score={score:.4f}")
        print("Title :", m.get("title"))
        print("Device:", m.get("device"))
        print("URL   :", m.get("url"))
        print("Text  :", m.get("preview"))
show(search("MacBook Air battery replacement", 5))


#1 score=0.8597
Title : MacBook Air Models A1237 and A1304 Battery Replacement
Device: MacBook Air Models A1237 and A1304
URL   : https://www.ifixit.com/Guide/MacBook+Air+Models+A1237+and+A1304+Battery+Replacement/848
Text  : TITLE: MacBook Air Models A1237 and A1304 Battery Replacement DEVICE: MacBook Air Models A1237 and A1304 SUMMARY: Don't pay Apple to replace your worn out... INTRO: Don't pay Apple to replace your worn out battery. Do it

#2 score=0.8528
Title : MacBook Air 11" Mid 2013 Battery Replacement
Device: MacBook Air 11" Mid 2013
URL   : https://www.ifixit.com/Guide/MacBook+Air+11-Inch+Mid+2013+Battery+Replacement/16840
Text  : TITLE: MacBook Air 11" Mid 2013 Battery Replacement DEVICE: MacBook Air 11" Mid 2013 SUMMARY: Use this guide to replace the battery. Note: If... INTRO: Use this guide to replace the battery. Note: If there is a thin plas

#3 score=0.8524
Title : MacBook Air 13" Mid 2011 Battery Replacement
Device: MacBook Air 13" Mid 2011
URL   : https://www.ifixi

Trying a few retrieval tests

In [ ]:
print("has model:", "model" in globals())
print("has index:", "index" in globals())
print("has meta :", "meta" in globals())

has model: True
has index: True
has meta : True


Top-k search + print

In [ ]:
import numpy as np

def search(query: str, top_k: int = 5):
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        m = meta[idx]
        results.append((float(score), m))
    return results

def pretty_print(results):
    for rank, (score, m) in enumerate(results, 1):
        print(f"\n#{rank} score={score:.4f}")
        print("Title:", m.get("title"))
        print("Device:", m.get("device"))
        print("URL:", m.get("url"))
        print("Preview:", m.get("preview"))

In [ ]:
def search_bge(query: str, top_k: int = 5):
    qtext = "Represent this sentence for searching relevant passages: " + query
    q = model.encode([qtext], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q, top_k)
    return [(float(s), meta[idx]) for s, idx in zip(scores[0], ids[0])]

pretty_print(search_bge("iPhone 11 screen replacement", 5))


#1 score=0.8305
Title: iPhone 11 Screen Replacement
Device: iPhone 11
URL: https://www.ifixit.com/Guide/iPhone+11+Screen+Replacement/135705
Preview: TITLE: iPhone 11 Screen Replacement DEVICE: iPhone 11 SUMMARY: If your iPhone 11 screen is cracked, not... INTRO: If your iPhone 11 screen is cracked, not responding to touch, or not showing a picture when powered on, us

#2 score=0.8286
Title: iPhone 11 Display Panel Replacement
Device: iPhone 11
URL: https://www.ifixit.com/Guide/iPhone+11+Display+Panel+Replacement/126236
Preview: TITLE: iPhone 11 Display Panel Replacement DEVICE: iPhone 11 SUMMARY: If your iPhone 11 screen is cracked, not... INTRO: If your iPhone 11 screen is cracked, not responding to touch, or not showing a picture when powered

#3 score=0.8270
Title: iPhone 11 Series Non-Genuine Screen Warning/Important Display Message - 100% Fix
Device: iPhone 11 Pro
URL: https://www.ifixit.com/Guide/iPhone+11+Series+Non-Genuine+Screen+Warning-Important+Display+Message+-+100%25+Fix/